# Proyecto Integrador: El Desafío del Robot Secuestrado (Kidnapped Robot Problem)

¡Bienvenido al proyecto final de graduación del curso! En este integrador nos enfrentaremos a uno de los problemas más difíciles en robótica autónoma y vehículos autónomos: **El Secuestro del Robot**.

### Escenario del Proyecto:
- El robot se mueve de forma autónoma dentro de un **laberinto cerrado** de $50 \times 50$ metros representado por una cuadrícula de ocupación (mapa binario de paredes).
- El robot tiene **colisiones físicas**: no puede atravesar paredes.
- Posee un sensor de rango tipo **LIDAR de 8 haces** que mide la distancia radial a los muros en 8 direcciones cardinales y ordinales.
- En la mitad de la simulación (Paso 15), **secuestraremos físicamente al robot**, teletransportándolo a una posición completamente lejana sin previo aviso.
- Programaremos un **Filtro de Partículas Adaptativo con Inyección de Ruido de Medición** para detectar el secuestro y auto-localizar al robot de manera brillante.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import time

# Estilo gráfico premium
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (9, 9)
plt.rcParams['font.size'] = 11

---
## 1. Definición del Laberinto y Sistema de Colisiones

Definiremos un mapa binario de $50 \times 50$ celdas donde el valor `1` representa un muro y `0` representa pasillo libre.

In [ ]:
map_size = 50
mapa = np.zeros((map_size, map_size), dtype=int)

# Dibujar paredes exteriores (bordes)
mapa[0, :] = 1
mapa[-1, :] = 1
mapa[:, 0] = 1
mapa[:, -1] = 1

# Agregar algunos obstáculos y pasillos internos estilo laberinto
mapa[10:15, 10:40] = 1
mapa[25:40, 15:20] = 1
mapa[20:35, 35:40] = 1

def es_libre(x, y):
    # Comprobar límites continuos y si la celda es libre
    ix, iy = int(x), int(y)
    if ix < 0 or ix >= map_size or iy < 0 or iy >= map_size:
        return False
    return mapa[iy, ix] == 0  # 0 es espacio libre

def dibujar_mapa(ax):
    # Grafica el mapa: 1 (paredes) en negro, 0 (libre) en blanco
    ax.imshow(mapa, cmap='binary', origin='lower', extent=[0, map_size, 0, map_size])

---
## 2. El Robot con Sensores LIDAR y Colisiones

El robot mide distancias a las paredes circundantes a lo largo de 8 rayos espaciados a $45^\circ$ cada uno.

In [ ]:
class LabyrinthRobot:
    def __init__(self, x, y, theta=0.0):
        self.x = x
        self.y = y
        self.theta = theta
        
        self.noise_forward = 0.3
        self.noise_turn = 0.04
        self.noise_sense = 1.2
        
    def move(self, turn, forward):
        # 1. Simular ruido de control
        theta_nuevo = (self.theta + turn + np.random.normal(0, self.noise_turn)) % (2 * np.pi)
        dist = forward + np.random.normal(0, self.noise_forward)
        
        # 2. Calcular nueva posición tentativa
        x_tentative = self.x + dist * np.cos(theta_nuevo)
        y_tentative = self.y + dist * np.sin(theta_nuevo)
        
        # 3. Control de colisiones: Solo actualizar posición si es libre de obstáculos
        if es_libre(x_tentative, y_tentative):
            self.x = x_tentative
            self.y = y_tentative
            self.theta = theta_nuevo
        else:
            # Colisión: Gira pero no avanza
            self.theta = theta_nuevo
            
    def sense_lidar(self):
        # Sensor LIDAR de 8 haces radiales
        haces = 8
        rangos = []
        
        for i in range(haces):
            angulo_haz = self.theta + i * (2 * np.pi / haces)
            
            # Lanzamiento del rayo por trazado de pasos hasta encontrar muro (Raymarching simple)
            step_size = 0.5
            dist = 0.0
            while dist < 40.0:  # Rango de visión de 40 metros máximo
                rx = self.x + dist * np.cos(angulo_haz)
                ry = self.y + dist * np.sin(angulo_haz)
                if not es_libre(rx, ry):
                    break
                dist += step_size
            
            # Agregar ruido al sensor
            rangos.append(dist + np.random.normal(0, self.noise_sense))
            
        return np.array(rangos)

---
## 3. El Filtro de Partículas Adaptativo con Inyección

Para recuperarnos de un secuestro, utilizaremos un monitoreo de la verosimilitud de la lectura del sensor. Si la verosimilitud promedio del sensor disminuye drásticamente, inyectaremos un **$15\%$ de partículas completamente nuevas** distribuidas al azar en cualquier área libre del laberinto.

In [ ]:
def crear_particulas_libres(M):
    # Genera partículas asegurando que caigan en zonas libres de obstáculos
    particulas = []
    while len(particulas) < M:
        px = np.random.rand() * map_size
        py = np.random.rand() * map_size
        if es_libre(px, py):
            particulas.append(LabyrinthRobot(px, py, theta=np.random.rand()*2*np.pi))
    return particulas

def predict_labyrinth(particulas, turn, forward):
    for p in particulas:
        # Propagar partículas respetando colisiones físicas contra el laberinto
        p.move(turn, forward)

def update_labyrinth(particulas, z, noise_sense=1.2):
    pesos = []
    for p in particulas:
        likelihood = 1.0
        # Evaluar la lectura LIDAR que vería la partícula en su estado hipotético
        # Desactivamos el ruido temporal para evaluar la PDF real
        ruido_original = p.noise_sense
        p.noise_sense = 0.0
        z_simulado = p.sense_lidar()
        p.noise_sense = ruido_original
        
        # Evaluar el Likelihood multivariable
        for i in range(8):
            prob = norm.pdf(z[i], loc=z_simulado[i], scale=noise_sense)
            likelihood *= prob
            
        pesos.append(likelihood)
        
    pesos = np.array(pesos) + 1e-300
    # Guardamos la verosimilitud media para monitorear secuestros
    verosimilitud_media = np.mean(pesos)
    pesos /= np.sum(pesos)
    return pesos, verosimilitud_media

def resample_adaptive(particulas, pesos, verosimilitud_media, w_slow, w_fast, M):
    """
    Remuestreo adaptativo con inyección de partículas aleatorias ante secuestros.
    """
    # 1. Actualizar promedios a largo y corto plazo de verosimilitud
    alpha_slow = 0.05
    alpha_fast = 0.2
    
    w_slow += alpha_slow * (verosimilitud_media - w_slow)
    w_fast += alpha_fast * (verosimilitud_media - w_fast)
    
    # 2. Calcular tasa de inyección aleatoria
    p_random = max(0.0, 1.0 - (w_fast / (w_slow + 1e-300)))
    # Limitar para evitar inestabilidad extrema
    p_random = min(0.35, p_random)
    
    num_aleatorios = int(M * p_random)
    num_resampling = M - num_aleatorios
    
    nuevas_particulas = []
    
    # A) Remuestreo sistemático de partículas sobrevivientes
    if num_resampling > 0:
        index = int(np.random.rand() * M)
        beta = 0.0
        max_w = np.max(pesos)
        for _ in range(num_resampling):
            beta += np.random.rand() * 2.0 * max_w
            while beta > pesos[index]:
                beta -= pesos[index]
                index = (index + 1) % M
            copia = LabyrinthRobot(particulas[index].x, particulas[index].y, particulas[index].theta)
            nuevas_particulas.append(copia)
            
    # B) Inyección activa de nuevas hipótesis
    if num_aleatorios > 0:
        nuevas_particulas.extend(crear_particulas_libres(num_aleatorios))
        
    return nuevas_particulas, w_slow, w_fast, p_random

def estimar_estado(particulas, pesos):
    x_est = 0.0
    y_est = 0.0
    for i, p in enumerate(particulas):
        x_est += p.x * pesos[i]
        y_est += p.y * pesos[i]
    return x_est, y_est

---
## 4. Bucle Interactivo con Secuestro

- El robot inicia físicamente en $(8.0, 8.0)$.
- En el **Paso 15**, secuestraremos al robot teletransportándolo instantáneamente a la zona libre en $(42.0, 42.0)$.

In [ ]:
np.random.seed(15)
M = 800

robot = LabyrinthRobot(x=8.0, y=8.0, theta=0.0)
particulas = crear_particulas_libres(M)

# Inicializadores de secuestro
w_slow = 1e-10
w_fast = 1e-10

print("🟢 INICIANDO PROYECTO INTEGRADOR...")

for paso in range(1, 26):
    # 1. Comandos de control predefinidos para explorar el laberinto
    turn = 0.1
    forward = 2.5
    
    # --- EVENTO DE SECUESTRO EN EL PASO 15 ---
    if paso == 15:
        print("\n🚨 🚨 🚨 ¡ALERTA: EL ROBOT HA SIDO SECUESTRADO! Teletransportando a (42.0, 42.0) 🚨 🚨 🚨\n")
        robot.x = 42.0
        robot.y = 42.0
        robot.theta = np.pi
        
    # Mover el robot real y sensar
    robot.move(turn, forward)
    z = robot.sense_lidar()
    
    # 2. FILTRO DE PARTÍCULAS
    predict_labyrinth(particulas, turn, forward)
    pesos, verosimilitud_media = update_labyrinth(particulas, z, noise_sense=1.2)
    x_est, y_est = estimar_estado(particulas, pesos)
    
    # Remuestrear de forma adaptativa
    particulas, w_slow, w_fast, p_random = resample_adaptive(
        particulas, pesos, verosimilitud_media, w_slow, w_fast, M
    )
    
    # Graficar instantes significativos
    # Paso 1: Localización uniforme
    # Paso 10: Convergencia inicial impecable
    # Paso 15: Confusión total tras el secuestro
    # Paso 18: Redescubrimiento de hipótesis e inyección de partículas
    # Paso 25: Relocalización exitosa convergente
    if paso in [1, 10, 15, 18, 25]:
        fig, ax = plt.subplots(figsize=(7.5, 7.5))
        dibujar_mapa(ax)
        
        # Partículas
        coords_p = np.array([[p.x, p.y] for p in particulas])
        ax.scatter(coords_p[:, 0], coords_p[:, 1], color='#d62728', alpha=0.35, s=6, label='Partículas')
        
        # Robot real y estimado
        ax.scatter(robot.x, robot.y, color='lime', marker='o', s=180, edgecolors='black', label='Robot Real')
        ax.scatter(x_est, y_est, color='cyan', marker='X', s=180, edgecolors='black', label='Estimación')
        
        ax.set_title(f'Laberinto - Paso {paso} | Inyección: {p_random*100:.1f}%', pad=12)
        ax.set_xlim(0, map_size)
        ax.set_ylim(0, map_size)
        ax.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.95)
        plt.show()
        
    print(f"⏱️ Paso {paso:02d} | Real: ({robot.x:.2f}, {robot.y:.2f}) | Est: ({x_est:.2f}, {y_est:.2f}) | Inyección Activa: {p_random*100:.1f}%")
    time.sleep(0.3)

---
## 🎓 Conclusiones y Aprendizaje de Nivel Profesional

Mediante este integrador has presenciado cómo:
1. Las colisiones físicas e imperfecciones del mundo real deforman las campanas Gaussianas clásicas convirtiéndolas en distribuciones multimodales complejas e irregulares.
2. Monitorear los pesos de verosimilitud de las partículas nos da una indicación inmediata del nivel de 'confusión' de nuestro sistema inteligente.
3. La inyección determinista de nuevas hipótesis basadas en la verosimilitud de las lecturas actuales permite resolver un problema supuestamente catastrófico como es el **secuestro físico del sistema**, adaptando la nube dinámicamente y logrando la supervivencia y robustez requerida en la industria moderna.

¡Felicidades, has culminado el curso completo de 0 a Pro en Filtros Bayesianos y de Partículas!